# HyDE: generate the hypothetical documents (Colab, T4)

Qwen2.5-3B-Instruct writes 8 hypothetical documents per test query for SciFact, NFCorpus and ArguAna, with the paper's sampling settings (temperature 0.7, top-p 1, at most 512 tokens). The paper used InstructGPT, which is retired.

Output: `generations/<dataset>.jsonl.gz`, one line per query: `{"qid": ..., "docs": [8 texts]}`. Everything else in this folder runs from that cache on a CPU.

In [ ]:
!pip -q install vllm
!git clone -q -b 15-hyde https://github.com/kenichisutan/ir2-testbed
%cd ir2-testbed

In [ ]:
import gzip, json, sys
from pathlib import Path
import vllm
from vllm import LLM, SamplingParams

sys.path += [".", "pairs/15-hyde"]
import testbed
from prompts import PROMPTS

print("vllm", vllm.__version__)
llm = LLM("Qwen/Qwen2.5-3B-Instruct", dtype="half", max_model_len=2048, seed=0)
params = SamplingParams(n=8, temperature=0.7, top_p=1.0, max_tokens=512, seed=0)
out = Path("pairs/15-hyde/generations")
out.mkdir(exist_ok=True)

In [ ]:
for name in testbed.DATASETS:
    corpus, queries, qrels = testbed.load(name)
    chats = [[{"role": "user", "content": PROMPTS[name].format(text)}] for text in queries.values()]
    results = llm.chat(chats, params)
    with gzip.open(out / f"{name}.jsonl.gz", "wt") as f:
        for qid, r in zip(queries, results):
            f.write(json.dumps({"qid": qid, "docs": [o.text.strip() for o in r.outputs]}) + "\n")
    print(name, len(results), "queries")

In [ ]:
!cd pairs/15-hyde && zip -q -r /content/generations.zip generations
from google.colab import files
files.download("/content/generations.zip")